# Day 008 · 协方差与相关性
**Covariance & Correlation** · 阶段 P1 · 量化基础

> 你以为买了三只不同的股票就分散了?如果三只都是新能源车赛道,你只买了一个篮子。这节课讲懂协方差、皮尔逊/斯皮尔曼相关、协方差矩阵,以及极端行情下相关性骤升的真相。

---

**课件生成日期:** 2026-05-01  ·  **建议学习时长:** 18 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [ ]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


## 实操:协方差矩阵 + 真实组合的伪分散陷阱

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [ ]:
# day_008_correlation.py — 比亚迪/宁德/特斯拉/中石油/黄金 真实相关矩阵
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

tickers = {
    '比亚迪':      '002594.SZ',
    '宁德时代':    '300750.SZ',
    '特斯拉':      'TSLA',
    '中国石油':    '601857.SS',
    '黄金 ETF':    'GLD',
}
df = yf.download(list(tickers.values()), period='2y', auto_adjust=True, progress=False)['Close']
df.columns = list(tickers.keys())
df = df.dropna()
ret = df.pct_change().dropna()

pearson = ret.corr()
print('=== 皮尔逊相关性矩阵 ===')
print(pearson.round(2).to_string())

spearman = ret.corr(method='spearman')
print('\n=== 斯皮尔曼相关性矩阵 ===')
print(spearman.round(2).to_string())

vol = ret.std() * np.sqrt(252)
cov_year = ret.cov() * 252

def portfolio_vol(weights, cov):
    return float(np.sqrt(weights.T @ cov @ weights))

wA = np.array([1/3, 1/3, 1/3, 0, 0])
vA = portfolio_vol(wA, cov_year.values)
wB = np.array([1/3, 0, 0, 1/3, 1/3])
vB = portfolio_vol(wB, cov_year.values)

print('\n=== 单资产年化波动 ===')
for n, v in vol.items():
    print(f'  {n}: {v*100:.1f}%')
print(f'\n组合 A 三只新能源车: 年化波动 {vA*100:.1f}% — 几乎没分散')
print(f'组合 B 比亚迪+中石油+黄金: 年化波动 {vB*100:.1f}% — 真分散')
print(f'差距 {(vA-vB)*100:.1f} 个百分点 = {(vA-vB)/vB*100:.0f}% 多余风险')

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pearson, cmap='RdYlGn_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(pearson))); ax.set_yticks(range(len(pearson)))
ax.set_xticklabels(pearson.columns, rotation=30, ha='right')
ax.set_yticklabels(pearson.index)
for i in range(len(pearson)):
    for j in range(len(pearson)):
        ax.text(j, i, f'{pearson.iloc[i,j]:.2f}', ha='center', va='center', color='black')
plt.colorbar(im); plt.title('皮尔逊相关性热力图(2 年日收益)')
plt.tight_layout(); plt.savefig('day008_corr_heatmap.png', dpi=120)

## 下一节预告

**Day 009 · 线性回归与最小二乘** (OLS & Linear Regression)


